In [1]:
!pip install -U scikit-learn
!pip install -U keras

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 24.7 MB/s  0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 14.9 MB/s  0:00:00
  Attempting uninstall: keras
    Found existing installation: keras 3.13.2
    Uninstalling keras-3.13.2:
      Successfully uninstalled keras-3.13.2


In [2]:
import tensorflow as tf
print ("Tensorflow version: ", tf.__version__)
from tensorflow import keras
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
import numpy as np

Tensorflow version:  2.21.0


In [4]:
cifar10 = tf.keras.datasets.cifar10
(x_train, Y_train), (x_test,Y_test) = cifar10.load_data()
print(x_train.shape, Y_train.shape, x_test.shape, Y_test.shape)

x_train = x_train/255.0
x_test = x_test/255.0

x_train = x_train.reshape(x_train.shape[0], -1)
x_test = x_test.reshape(x_test.shape[0], -1)

knn_1 = KNeighborsClassifier(n_neighbors=1)
knn_3 = KNeighborsClassifier(n_neighbors=3)


  1875968/170498071 ━━━━━━━━━━━━━━━━━━━━ 55:31 20us/step

KeyboardInterrupt: 

In [ ]:
knn_1.fit(x_train, Y_train.ravel())
knn_3.fit(x_train, Y_train.ravel())

In [ ]:
Y_pred_1 = knn_1.predict(x_test)
Y_pred_3 = knn_3.predict(x_test)


accuracy_1 = accuracy_score(Y_test,Y_pred_1)
accuracy_3 = accuracy_score(Y_test, Y_pred_3)

print(f"Accuracy of 1-NN: {accuracy_1:.4f}")
print(f"Accuracy of 3-NN: {accuracy_3:.4f}")

In [ ]:
from sklearn.neighbors import NearestCentroid

centroid = NearestCentroid()
centroid.fit(x_train, Y_train.ravel())


In [ ]:
Y_pred_centroid = centroid.predict(x_test)

accuracy_centroid = accuracy_score(Y_test, Y_pred_centroid)

print(f"Accuracy of Nearest Centroid: {accuracy_centroid:.4f}")


In [ ]:
from hyperopt import hp, fmin, tpe, Trials

In [ ]:
# Define the hyperparameter space for KNN
knn_space = {
    'n_neighbors': hp.choice('n_neighbors', [1, 3, 5, 7, 9]),
    'metric': hp.choice('metric', ['euclidean', 'manhattan', 'chebyshev'])
}

# Define the objective function for KNN
def objective_knn(params):
    knn = KNeighborsClassifier(n_neighbors=params['n_neighbors'], metric=params['metric'])
    knn.fit(x_train, Y_train.ravel())
    Y_pred = knn.predict(x_test)
    accuracy = accuracy_score(Y_test, Y_pred)
    return -accuracy  # Negative because we want to minimize the loss

# Apply Hyperopt to optimize KNN
trials_knn = Trials()
best_params_knn = fmin(fn=objective_knn,
                       space=knn_space,
                       algo=tpe.suggest,
                       max_evals=50,
                       trials=trials_knn)

print("Best Hyperparameters for KNN:", best_params_knn)

# Train and evaluate KNN with the best hyperparameters
knn_optimized = KNeighborsClassifier(
    n_neighbors=best_params_knn['n_neighbors'],
    metric=best_params_knn['metric']
)
knn_optimized.fit(x_train, Y_train.ravel())
Y_pred_knn_optimized = knn_optimized.predict(x_test)

accuracy_knn_optimized = accuracy_score(Y_test, Y_pred_knn_optimized)
print(f"Accuracy of KNN after Hyperopt: {accuracy_knn_optimized:.4f}")



In [ ]:
# Define the hyperparameter space for the Nearest Centroid classifier
centroid_space = {
    'metric': hp.choice('metric', ['euclidean', 'manhattan']),
    'shrink_threshold': hp.uniform('shrink_threshold', 0.0, 1.0)
}

# Define the objective function for Nearest Centroid
def objective_centroid(params):
    centroid = NearestCentroid(
        metric=params['metric'],
        shrink_threshold=params['shrink_threshold']
    )
    centroid.fit(x_train, Y_train.ravel())
    Y_pred = centroid.predict(x_test)
    accuracy = accuracy_score(Y_test, Y_pred)
    return -accuracy  # Negative because we want to minimize the loss

# Apply Hyperopt to optimize Nearest Centroid
trials_centroid = Trials()
best_params_centroid = fmin(fn=objective_centroid,
                            space=centroid_space,
                            algo=tpe.suggest,
                            max_evals=50,
                            trials=trials_centroid)

print("Best Hyperparameters for Nearest Centroid:", best_params_centroid)

# Train and evaluate Nearest Centroid with the best hyperparameters
centroid_optimized = NearestCentroid(
    metric=best_params_centroid['metric'],
    shrink_threshold=best_params_centroid['shrink_threshold']
)
centroid_optimized.fit(x_train, Y_train.ravel())
Y_pred_centroid_optimized = centroid_optimized.predict(x_test)

accuracy_centroid_optimized = accuracy_score(Y_test, Y_pred_centroid_optimized)
print(f"Accuracy of Nearest Centroid after Hyperopt: {accuracy_centroid_optimized:.4f}")
